# Stage 37B: top-PF pseudo-private materialization

固定Stage 37A manifestのtraining 1,370 cutsとdesign 62 cutsだけに、target-safe top-PF proxyを計算します。20 cutsごとのchunkをDriveへ保存するため、中断後は同じNotebookを再実行すると完了chunkを再利用します。confirmation 120 wellsは開きません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir():
    subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else:
    subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
def run_checked(command):
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout,flush=True)
    if result.returncode:
        print(result.stderr,flush=True); raise RuntimeError(f'command failed: {command}')


## 固定入力

Stage 37Aのmanifest hashが設定値と一致しない場合は停止します。実行には時間がかかります。途中停止してもrun directoryを削除しないでください（Do not delete）。

In [ ]:
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
stage37a_run=artifact_dir/'stage37a_pseudo_private_manifest_v001'
assert (stage17a_run/'replay_predictions.parquet').is_file(),stage17a_run
assert (stage37a_run/'pseudo_private_manifest.parquet').is_file(),stage37a_run
source_summary=json.loads((stage37a_run/'summary.json').read_text())
assert source_summary['pseudo_private_manifest_sha256']=='a51c2776a63515e27aa1177ede37abebfd021dabea5fb93436d2635a97a4d166',source_summary
RUN_ID='stage37b_top_pf_materialization_v001'; run_dir=artifact_dir/RUN_ID
if not (run_dir/'summary.json').is_file():
    subprocess.run(['uv','run','rogii-pseudo-private-top-pf','--config','configs/experiment/stage37b_top_pf_materialization.yaml','--stage17a-run',str(stage17a_run),'--stage37a-run',str(stage37a_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID],cwd=repo_dir,check=True)
else:
    print('Reusing completed Stage 37B:',run_dir)
summary=json.loads((run_dir/'summary.json').read_text())
summary

In [ ]:
{key:summary[key] for key in ['stage37b_complete','promoted_to_stage38_retrieval_v2','stage37a_manifest_sha256','materialized_cuts','materialized_wells','prediction_rows','chunks','roles','confirmation_wells_materialized','confirmation_target_columns_read','gates','next_step']}

最後の辞書を共有してください。処理が中断した場合は同じNotebookをもう一度実行してください。`chunk ... reused`と表示され、残りから継続します。